# Machine Learning Practicals – Clean Exam Cheat Sheet
Keep this notebook in the same folder as your CSV datasets.

## Cell 0: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

## Cell 1: Load & Inspect Dataset (A1)

In [ ]:
df = pd.read_csv('students.csv')

print(df.head())
print(df.tail())
print(df.shape)
print(df.columns)

df.info()
print(df.describe())
print(df.isnull().sum())
print(df.duplicated().sum())

## Cell 2: Missing Values & Duplicates (A2 – A3)

In [ ]:
# Drop rows with missing values
df_clean = df.dropna()

# Fill numerical missing values
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Salary'] = df['Salary'].fillna(df['Salary'].median())

# Fill text missing values
df['City'] = df['City'].fillna(df['City'].mode()[0])

# Remove duplicate rows
df.drop_duplicates(inplace=True)

## Cell 3: Filter, Sort & Column Operations (A4 – A6)

In [ ]:
# Filter rows
high_scorers = df[df['Marks'] > 70]
filtered = df[(df['Marks'] > 70) & (df['Age'] > 20)]

# Sort rows
sorted_df = df.sort_values(by='Marks', ascending=False)

# Rename and drop columns
df.rename(columns={'Marks': 'Score'}, inplace=True)
df.drop('Age', axis=1, inplace=True)

# Clean text
df['Name'] = df['Name'].astype('string').str.strip().str.upper()
df['City'] = df['City'].astype('string').str.lower()
df['City'] = df['City'].str.replace('ahemdabad', 'ahmedabad', regex=False)

## Cell 4: Dates, Calculated Columns, GroupBy & Merge (A7 – A12)

In [ ]:
# Extract Year, Month, Day
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

# Create calculated columns
df['Total'] = df['Maths'] + df['Science']
df['Average'] = df['Total'] / 2
df['Result'] = np.where(df['Average'] >= 40, 'Pass', 'Fail')

# Group and aggregate
summary = df.groupby('City')['Score'].agg(['sum', 'mean', 'count', 'max', 'min'])
print(summary)

# Merge two CSV files
students = pd.read_csv('students.csv')
results = pd.read_csv('results.csv')
merged = pd.merge(students, results, on='StudentID', how='inner')

# Export cleaned CSV
df.to_csv('processed_data.csv', index=False)

## Cell 5: Regression Evaluation Helper Function

In [ ]:
def evaluate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

## Cell 6: Question 1 – Linear Regression (Student Score)

In [ ]:
st_perf = pd.read_csv('student_performance_records.csv')

X = st_perf.drop(columns=['final_exam_score'])
y = st_perf['final_exam_score']

# 1. Model with ALL features
X_train_all, X_test_all, y_train, y_test = train_test_split(
    X, y, test_size=0.05, random_state=50
)

model_all = LinearRegression()
model_all.fit(X_train_all, y_train)
pred_all = model_all.predict(X_test_all)

mae_all, rmse_all, r2_all = evaluate_metrics(y_test, pred_all)
print(f'All Features -> MAE: {mae_all:.2f}, RMSE: {rmse_all:.2f}, R2: {r2_all:.2f}')

# 2. Model with SELECTED features
subset_features = ['study_hours_per_week', 'attendance_percentage', 'internal_exam_score']
X_subset = st_perf[subset_features]

X_train, X_test, _, _ = train_test_split(
    X_subset, y, test_size=0.05, random_state=50
)

model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae, rmse, r2 = evaluate_metrics(y_test, pred)
print(f'Selected Features -> MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.2f}')

## Cell 7: Question 2 – Linear Regression (House Price)

In [ ]:
house = pd.read_csv('house_selling_prices.csv')

X = house.drop(columns=['price_lakhs'])
y = house['price_lakhs']

# 1. With all features
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=50
)

house_model = LinearRegression()
house_model.fit(X_train, y_train)
house_pred = house_model.predict(X_test)

mae, rmse, r2 = evaluate_metrics(y_test, house_pred)
print(f'All Features -> MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.2f}')

# 2. Without area_sqft
X_subset = house.drop(columns=['price_lakhs', 'area_sqft'])

X_train_s, X_test_s, _, _ = train_test_split(
    X_subset, y, test_size=0.20, random_state=50
)

house_model_sub = LinearRegression()
house_model_sub.fit(X_train_s, y_train)
house_pred_sub = house_model_sub.predict(X_test_s)

mae, rmse, r2 = evaluate_metrics(y_test, house_pred_sub)
print(f'Without area_sqft -> MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.2f}')

## Cell 8: Question 3 – Logistic Regression (Student Pass/Fail)

In [ ]:
st_pf = pd.read_csv('student_pass_fail_records.csv')

X = st_pf.drop(columns=['pass'])
y = st_pf['pass']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# Get probability for class 1
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Evaluate at Threshold 0.50
y_pred_50 = (y_prob >= 0.50).astype(int)
print('--- Threshold 0.50 ---')
print('Accuracy :', accuracy_score(y_test, y_pred_50))
print('Precision:', precision_score(y_test, y_pred_50))
print('Recall   :', recall_score(y_test, y_pred_50))
print('F1-score :', f1_score(y_test, y_pred_50))
print(confusion_matrix(y_test, y_pred_50))

# Evaluate at Threshold 0.70
y_pred_70 = (y_prob >= 0.70).astype(int)
print('\n--- Threshold 0.70 ---')
print('Accuracy :', accuracy_score(y_test, y_pred_70))
print('Precision:', precision_score(y_test, y_pred_70))
print('Recall   :', recall_score(y_test, y_pred_70))
print('F1-score :', f1_score(y_test, y_pred_70))
print(confusion_matrix(y_test, y_pred_70))

## Cell 9: Question 4 – Logistic Regression (Customer Churn)

In [ ]:
df_cust = pd.read_csv('telecom_customer_churn.csv')

X = df_cust.drop(columns=['churn'])
y = df_cust['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred_50 = (y_prob >= 0.50).astype(int)

print(classification_report(y_test, y_pred_50, target_names=['Stay', 'Churn'], zero_division=0))

# Compare multiple thresholds
threshold_results = []
for thresh in [0.30, 0.40, 0.50, 0.60, 0.70]:
    y_pred = (y_prob >= thresh).astype(int)
    threshold_results.append({
        'Threshold': thresh,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-score': f1_score(y_test, y_pred, zero_division=0)
    })

print(pd.DataFrame(threshold_results))

## Cell 10: Question 5 – KNN (Student Performance)

In [ ]:
df = pd.read_csv('student_performance_classification.csv')

X = df.drop(columns=['performance_category'])
y = df['performance_category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale features (Required for KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Test different K values
results = []
for k in [1, 3, 5, 9, 15]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)

    results.append({
        'K': k,
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred, average='weighted')
    })

print(pd.DataFrame(results))

## Cell 11: Question 6 – KNN (With vs. Without Scaling)

In [ ]:
df = pd.read_csv('ecommerce_customer_segments.csv')

X = df.drop(columns=['customer_segment'])
y = df['customer_segment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 1. KNN WITHOUT Scaling
knn_no_scale = KNeighborsClassifier(n_neighbors=5)
knn_no_scale.fit(X_train, y_train)
pred_no_scale = knn_no_scale.predict(X_test)

acc_no_scale = accuracy_score(y_test, pred_no_scale) * 100
print(f'Without Scaling Accuracy: {acc_no_scale:.2f}%')

# 2. KNN WITH Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
pred_scaled = knn_scaled.predict(X_test_scaled)

acc_scaled = accuracy_score(y_test, pred_scaled) * 100
print(f'With Scaling Accuracy   : {acc_scaled:.2f}%')

## Cell 12: Question 7 – Decision Tree (Loan Approval)

In [ ]:
df = pd.read_csv('loan_applications.csv')

features = [
    'age', 'annual_income', 'credit_score', 'employment_years',
    'existing_loans', 'debt_to_income_ratio', 'previous_default', 'loan_amount'
]
X = df[features].copy()

# Encode text target (Approved / Rejected -> 0 / 1)
encoder = LabelEncoder()
y = encoder.fit_transform(df['loan_status'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Compare different max_depth values
results = []
for depth in [2, 4, 6, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train)) * 100
    test_acc = accuracy_score(y_test, model.predict(X_test)) * 100

    results.append({
        'max_depth': depth,
        'Training Accuracy': train_acc,
        'Testing Accuracy': test_acc,
        'Accuracy Gap': train_acc - test_acc
    })

results_df = pd.DataFrame(results)
print(results_df.round(2))

# Pick best depth and train final model
best_row = results_df.loc[results_df['Testing Accuracy'].idxmax()]
best_depth = None if pd.isna(best_row['max_depth']) else int(best_row['max_depth'])

final_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
final_model.fit(X_train, y_train)
final_pred = final_model.predict(X_test)

print('\nBest max_depth:', best_depth)
print('Final Accuracy:', round(accuracy_score(y_test, final_pred) * 100, 2), '%')

## Cell 13: Question 8 – Decision Tree (Employee Attrition)

In [ ]:
df = pd.read_csv('employee_attrition_records.csv')

features = [
    'age', 'years_at_company', 'monthly_income', 'overtime_hours',
    'job_satisfaction', 'promotion_years', 'work_life_balance', 'distance_from_home_km'
]
X = df[features]

# Encode text target (No / Yes -> 0 / 1)
encoder = LabelEncoder()
y = encoder.fit_transform(df['attrition'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Compare different max_depth values
results = []
for depth in [3, 5, 10]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    test_pred = model.predict(X_test)

    results.append({
        'max_depth': depth,
        'Training Accuracy': accuracy_score(y_train, model.predict(X_train)),
        'Testing Accuracy': accuracy_score(y_test, test_pred),
        'Precision': precision_score(y_test, test_pred, zero_division=0),
        'Recall': recall_score(y_test, test_pred, zero_division=0),
        'F1 Score': f1_score(y_test, test_pred, zero_division=0)
    })

results_df = pd.DataFrame(results)
print(results_df.round(3))

# Pick best depth and train final model
best_row = results_df.loc[results_df['Testing Accuracy'].idxmax()]
best_depth = int(best_row['max_depth'])

final_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
final_model.fit(X_train, y_train)
final_pred = final_model.predict(X_test)

print(f'\nBest max_depth: {best_depth}')
print(f'Final Accuracy: {accuracy_score(y_test, final_pred) * 100:.2f}%')